In [1]:
import pandas as pd

In [2]:
class DatosHidrologicos:
    """Contenedor de datos del pipeline hidrológico."""
    def __init__(self):
        self.df_topo:  pd.DataFrame = pd.DataFrame()
        self.df_final: pd.DataFrame = pd.DataFrame()

In [3]:
# Extraemos los datos del csv
class TopografiaExtractor:
    """Lee y normaliza el CSV topográfico exportado desde QGIS/IGN.
    Fuente: Instituto Geográfico Nacional (IGN). Modelo Digital del Terreno.
    Procesado con QGIS para el tramo Cañones del Sil (20 km).
    """

    def __init__(self, ruta: str):
        self.ruta = ruta

    def extraer(self) -> pd.DataFrame:
        print('Cargando perfil topográfico del río Sil (IGN/QGIS)')
        df = pd.read_csv(self.ruta)[['distance', 'angle', 'SAMPLE_1']]
        df.columns = ['Distancia_m', 'Angulo', 'Elevacion_m']
        print(f'Correcto => {len(df):,} puntos topográficos cargados.')
        return df


class CaudalesCEDEXExtractor:
    """Lee el histórico mensual de caudales del CEDEX y calcula medias estacionales.
    Fuente: CEDEX (Centro de Estudios y Experimentación de Obras Públicas).
    Anuario de Aforos. Estación 1018 — Río Sil.
    https://ceh.cedex.es/anuarioaforos/
    """

    # Definición de estaciones del año
    ESTACIONES = {
        'Caudal_Invierno':  [12, 1, 2],
        'Caudal_Primavera': [3, 4, 5],
        'Caudal_Verano':    [6, 7, 8],
        'Caudal_Otono':     [9, 10, 11],
    }

    def __init__(self, ruta: str, estacion_id: int = 1018):
        self.ruta        = ruta
        self.estacion_id = estacion_id

    def extraer_media_mensual(self) -> pd.Series:
        print(f'Procesando histórico CEDEX (Estación {self.estacion_id})...')
        df = pd.read_csv(self.ruta, encoding='latin1', sep=';', on_bad_lines='skip')
        df.columns = df.columns.str.strip().str.lower()

        df_est = df[df['indroea'] == self.estacion_id].copy()
        df_est['mes'] = df_est['anomes'].astype(str).str[-2:].astype(int)

        media = df_est.groupby('mes')['qmedmes'].mean()
        print(f' Correcto => {len(df_est):,} registros históricos procesados.')
        return media

    def calcular_estacionales(self, media_mensual: pd.Series) -> dict:
        """Calcula la media de caudal por estación del año."""
        return {
            nombre: media_mensual[meses].mean()
            for nombre, meses in self.ESTACIONES.items()
        }

In [4]:
# Procesamos los datos
class UnidorHidrologico:
    """Une el DataFrame topográfico con los caudales estacionales calculados."""

    @staticmethod
    def unir(df_topo: pd.DataFrame, caudales: dict) -> pd.DataFrame:
        print('Uniendo topografía y caudales estacionales')
        df = df_topo.copy()
        for nombre, valor in caudales.items():
            df[nombre] = valor
        # Q_medio global — columna de conveniencia para el Algoritmo Genético
        df['Q_medio'] = df[list(caudales.keys())].mean(axis=1)
        return df


# Exportamos el resultado en un csv
class CSVHidroExporter:
    """Guarda el dataset final en formato CSV."""

    @staticmethod
    def exportar(df: pd.DataFrame, ruta: str):
        df.to_csv(ruta, index=False)
        print(f'Dataset guardado → {ruta}')
        print(f' Correcto => {len(df):,} filas × {len(df.columns)} columnas.')
        print(f'   Columnas: {list(df.columns)}')

In [5]:
# Orquestador principal
def generar_dataset_hidrologico(
        ruta_topo:    str,
        ruta_mensual: str,
        ruta_salida:  str,
        estacion_id:  int = 1018,
) -> pd.DataFrame:
    print('=' * 55)
    print('PIPELINE HIDROLÓGICO')
    print('=' * 55)

    # Inicializamos el contenedor DTO
    contenedor = DatosHidrologicos()

    # Extraemos la topografía y la guardamos en el contenedor
    contenedor.df_topo = TopografiaExtractor(ruta_topo).extraer()

    # Extraemos y procesamos los caudales del CEDEX
    extractor_q        = CaudalesCEDEXExtractor(ruta_mensual, estacion_id)
    media              = extractor_q.extraer_media_mensual()
    caudales           = extractor_q.calcular_estacionales(media)

    # Unimos los datos usando las propiedades del contenedor
    contenedor.df_final = UnidorHidrologico.unir(contenedor.df_topo, caudales)

    # Exportamos el DataFrame final almacenado en el DTO
    CSVHidroExporter.exportar(contenedor.df_final, ruta_salida)

    return contenedor.df_final


# Ejecutamos
df = generar_dataset_hidrologico(
    ruta_topo    = '/content/drive/MyDrive/TFG - UAX/CSV/Topografia_Rio_sil.csv',
    ruta_mensual = '/content/drive/MyDrive/TFG - UAX/CSV/mensual_a.csv',
    ruta_salida  = '/content/drive/MyDrive/TFG - UAX/CSV/Sil_Dataset_Completo.csv',
    estacion_id  = 1018,
)

PIPELINE HIDROLÓGICO
Cargando perfil topográfico del río Sil (IGN/QGIS)
Correcto => 2,005 puntos topográficos cargados.
Procesando histórico CEDEX (Estación 1018)...
 Correcto => 390 registros históricos procesados.
Uniendo topografía y caudales estacionales
Dataset guardado → /content/drive/MyDrive/TFG - UAX/CSV/Sil_Dataset_Completo.csv
 Correcto => 2,005 filas × 8 columnas.
   Columnas: ['Distancia_m', 'Angulo', 'Elevacion_m', 'Caudal_Invierno', 'Caudal_Primavera', 'Caudal_Verano', 'Caudal_Otono', 'Q_medio']
